In [1]:
# Importing packages
import numpy as np
import numpy_financial as npf
import pandas as pd
from prepay_amort import get_cash_flows
from dateutil.relativedelta import relativedelta
from tqdm import tqdm
from datetime import date
from utils import VectorHandler, normalize_next_payment_date

In [2]:
CPR = "3.66 4.57 5.48 6.39 7.3 8.2 9.11 10.02 11.29 11.87 12.33 12.66 12.87 12.95 12.91 12.74 12.44 12.02 11.48 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5 11.5"
# 1x
# CDR = "0 0 0 0 0 2.37 4.7 6.97 8.75 9.08 9.41 9.74 10.06 10.39 10.71 11.04 11.36 11.68 12 12.32 12.64 12.96 13.28 13.59 13.91 14.22 14.53 14.84 15.15 15.46 15.77 16.08 16.38 16.69 16.99 17.3 17.6 17.9 18.2 18.5 18.8 19.09 19.39 19.68 19.98 20.27 20.56 20.85 21.14 21.53 21.53"
# 0.5x
# CDR = "0.00 0.00 0.00 0.00 0.00 1.19 2.35 3.48 4.38 4.54 4.71 4.87 5.03 5.20 5.36 5.52 5.68 5.84 6.00 6.16 6.32 6.48 6.64 6.79 6.96 7.11 7.26 7.42 7.58 7.73 7.88 8.04 8.19 8.35 8.49 8.65 8.80 8.95 9.10 9.25 9.40 9.54 9.70 9.84 9.99 10.13 10.28 10.43 10.57 10.77 10.77"
CDR = "0.0000 0.0000 0.0000 0.0000 0.0000 2.2157 4.3897 6.5316 8.2109 8.5219 8.8329 9.1439 9.4557 9.7683 10.0809 10.3943 10.7086 11.0229 11.3381 11.6533 11.9694 12.2856 12.6027 12.9207 13.2388 13.5578 13.8769 14.1969 14.5170 14.8380 15.1591 15.4812 15.8034 16.1265 16.4498 16.7740 17.0984 17.4237 17.7492 18.0756 18.4022 18.7297 19.0574 19.3860 19.7397 19.7397 19.7397 19.7397 19.7397 19.7397"
Severity = "70"
COUPON_MULT = "1.000 0.987 0.967 0.956 0.948 0.942 0.937 0.932 0.928 0.925 0.922 0.920 0.917 0.915 0.913 0.911 0.909 0.907 0.906 0.904 0.903 0.901 0.900 0.899 0.898 0.897 0.895 0.894 0.893 0.892 0.891 0.891 0.890 0.889 0.888 0.887 0.886 0.886 0.885 0.884 0.883 0.883 0.882 0.881 0.881 0.880 0.880 0.880 0.880"
TARGET_YEAR = 2024
TARGET_MONTH = 11

# Load
loan_dataset = pd.read_excel('Yields and Prices.xlsx')
loan_dataset = loan_dataset[loan_dataset['ASTAT'] == 0]
print(max(loan_dataset['ANXDTDT']))

# Convert dates
date_columns = ['ABKDTDT', 'ANXDTDT', 'ACODTDT', 'AUD3DT', 'AMTDTDT']
for col in date_columns:
    loan_dataset[col] = pd.to_datetime(loan_dataset[col])

# Add vectors
loan_dataset['CPR'] = CPR
loan_dataset['CDR'] = CDR
loan_dataset['Severity'] = Severity
loan_dataset['COUPON_MULT'] = COUPON_MULT

# Normalize next payment dates
loan_dataset['ANXDTDT'] = loan_dataset['ANXDTDT'].apply(
    lambda x: normalize_next_payment_date(x, TARGET_YEAR, TARGET_MONTH)
)
print(max(loan_dataset['ANXDTDT']))

loan_dataset.head()

2029-08-28 00:00:00
2024-11-30 00:00:00


,ABKDTDT,ASTAT,AACCT,APOOL,ANETBAL,AIACR,AUSC1,AOTRM,REMPMTS,MONTHBOARD,...,VLGLTV,CISTAT,VLDGIR,VLPGIR,AUD1D,AUD1DT,CPR,CDR,Severity,COUPON_MULT
0,2024-10-31,0,22362016,2820,10078.42,52.31,F05,54,42,202310,...,139.78,OR,25.68,4.92,2,10/02/23,3.66 4.57 5.48 6.39 7.3 8.2 9.11 10.02 11.29 1...,0.0000 0.0000 0.0000 0.0000 0.0000 2.2157 4.38...,70,1.000 0.987 0.967 0.956 0.948 0.942 0.937 0.93...
1,2024-10-31,0,22369755,2820,17598.22,125.21,I01,72,59,202310,...,93.24,KY,51.72,5.90,5,10/05/23,3.66 4.57 5.48 6.39 7.3 8.2 9.11 10.02 11.29 1...,0.0000 0.0000 0.0000 0.0000 0.0000 2.2157 4.38...,70,1.000 0.987 0.967 0.956 0.948 0.942 0.937 0.93...
2,2024-10-31,0,22447346,2820,17772.02,257.67,I01,72,60,202310,...,117.18,NC,41.14,12.53,20,10/20/23,3.66 4.57 5.48 6.39 7.3 8.2 9.11 10.02 11.29 1...,0.0000 0.0000 0.0000 0.0000 0.0000 2.2157 4.38...,70,1.000 0.987 0.967 0.956 0.948 0.942 0.937 0.93...
3,2024-10-31,0,22456313,2820,30144.45,787.03,F03,78,67,202310,...,140.99,TX,38.23,10.16,13,10/13/23,3.66 4.57 5.48 6.39 7.3 8.2 9.11 10.02 11.29 1...,0.0000 0.0000 0.0000 0.0000 0.0000 2.2157 4.38...,70,1.000 0.987 0.967 0.956 0.948 0.942 0.937 0.93...
4,2024-10-31,0,22457840,2820,18711.89,220.91,I01,66,53,202310,...,113.95,LA,18.41,7.98,6,10/06/23,3.66 4.57 5.48 6.39 7.3 8.2 9.11 10.02 11.29 1...,0.0000 0.0000 0.0000 0.0000 0.0000 2.2157 4.38...,70,1.000 0.987 0.967 0.956 0.948 0.942 0.937 0.93...


In [3]:
indx = 0
xddf, _, _ = get_cash_flows(
    apmt1=loan_dataset['APMT1'][indx],
    rempmts=loan_dataset['AOTRM'][indx],
    original_balance=loan_dataset['AOFIN'][indx], 
    unpaid_balance=loan_dataset['AOFIN'][indx], 
    interest_rate=loan_dataset['ARATE'][indx], 
    original_term=loan_dataset['AOTRM'][indx], 
    calculation_start_date=loan_dataset['ACODTDT'][indx],
    next_payment_date=loan_dataset['AUD3DT'][indx], 
    cpr=loan_dataset['CPR'][indx], 
    cdr=loan_dataset['CDR'][indx], 
    severity=loan_dataset['Severity'][indx],
    price=loan_dataset['PRICE'][indx],
    arpay=loan_dataset['ARPAY'][indx],
    xirr_calc=True,
    output=True,
    coupon_mult=loan_dataset['COUPON_MULT'][indx],
    reference_date=loan_dataset['ABKDTDT'][indx]  # Add this line
)
xddf.head()

,Date,UPB,Performing UPB,Gross Charge Offs,Net Loss,Recoveries,Scheduled Principal,Voluntary Prepayments,Principal Cash Flow,Interest Cash Flow,Total Cash Flow
0,2023-07-31,11182.840000,11182.840000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
1,2023-09-14,11011.059681,11011.059681,0.0,0.0,0.0,128.774348,43.005970,171.780319,286.444847,458.225165
2,2023-10-14,10829.554911,10829.554911,0.0,0.0,0.0,130.523479,50.981291,181.504770,184.219704,365.724474
3,2023-11-14,10638.660904,10638.660904,0.0,0.0,0.0,132.190776,58.703232,190.894007,185.092759,375.986767
4,2023-12-14,10438.741180,10419.268362,0.0,0.0,0.0,133.771482,66.148242,199.919724,174.492110,374.411834


In [4]:
xddf.tail()

,Date,UPB,Performing UPB,Gross Charge Offs,Net Loss,Recoveries,Scheduled Principal,Voluntary Prepayments,Principal Cash Flow,Interest Cash Flow,Total Cash Flow
50,2027-10-14,338.340456,332.196966,8.224953,5.757467,2.467486,102.874770,3.532725,108.874981,6.771379,115.646360
51,2027-11-14,228.263768,224.119020,6.143490,4.300443,1.843047,101.547972,2.385226,105.776245,5.226359,111.002604
52,2027-12-14,122.673927,120.446449,4.144749,2.901324,1.243425,100.160598,1.284495,102.688517,3.412258,106.100775
53,2028-01-14,21.624834,21.232176,2.227478,1.559235,0.668243,98.590531,0.231084,99.489859,1.894949,101.384808
54,2028-02-14,0.000000,0.000000,0.392658,0.274860,0.117797,21.349973,0.000000,21.467771,0.334040,21.801810


In [5]:
xddf.to_excel('amort_loan1_origination.xlsx')

In [6]:
i = 0
print(loan_dataset['AOTRM'][i], loan_dataset['REMPMTS'][i], loan_dataset['AOTRM'][i] - loan_dataset['REMPMTS'][i])
xddf, _, _ = get_cash_flows(
            apmt1=loan_dataset['APMT1'][i],
            rempmts=loan_dataset['REMPMTS'][i],
            original_balance=loan_dataset['AOFIN'][i], 
            unpaid_balance=loan_dataset['ANETBAL'][i],
            interest_rate=loan_dataset['ARATE'][i],
            original_term=loan_dataset['AOTRM'][i],
            calculation_start_date=loan_dataset['ABKDTDT'][i],
            next_payment_date=loan_dataset['ANXDTDT'][i],
            cpr=loan_dataset['CPR'][i],
            cdr=loan_dataset['CDR'][i],
            severity=loan_dataset['Severity'][i],
            price=loan_dataset['PRICE'][i],
            output=True,
            arpay=loan_dataset['ARPAY'][i],
            maturity_date=loan_dataset['AMTDTDT'][i],
            contract_date=loan_dataset['ACODTDT'][i],
            coupon_mult=loan_dataset['COUPON_MULT'][i]
        )
xddf[35:44]

54 42 12


,Date,UPB,Performing UPB,Gross Charge Offs,Net Loss,Recoveries,Scheduled Principal,Voluntary Prepayments,Principal Cash Flow,Interest Cash Flow,Total Cash Flow
35,2027-09-14,1936.488610,1901.326400,37.868367,26.507857,11.360510,90.976938,20.190890,122.528338,32.215190,154.743527
36,2027-10-14,1792.703431,1760.152031,35.162210,24.613547,10.548663,89.930934,18.692035,119.171632,28.948085,148.119717
37,2027-11-14,1654.010376,1623.977326,32.551399,22.785980,9.765420,88.895394,17.246261,115.907075,27.691966,143.599040
38,2027-12-14,1520.255322,1492.650958,30.033050,21.023135,9.009915,87.870044,15.851960,112.731919,24.725388,137.457307
39,2028-01-14,1391.288806,1366.026180,27.604364,19.323055,8.281309,86.854578,14.507574,109.643461,23.483448,133.126909
40,2028-02-14,1266.965930,1243.960723,25.262627,17.683839,7.578788,85.848655,13.211594,106.639037,21.491297,128.130334
41,2028-03-14,1147.146280,1126.316724,23.005207,16.103645,6.901562,84.851888,11.962555,103.716005,18.308239,122.024244
42,2028-04-14,1031.693861,0.000000,20.829556,14.580689,0.000000,83.863822,10.759040,0.000000,0.000000,0.000000
43,2028-05-14,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [7]:
xddf.to_excel('amort_loan1_october.xlsx')

In [8]:
yields = []
priceNa = 0
for index, row in tqdm(loan_dataset.iterrows()):
    if pd.notna(row['PRICE']):
        loan_yield = get_cash_flows(
            apmt1=row['APMT1'],
            rempmts=row['AOTRM'],
            original_balance=row['AOFIN'], 
            unpaid_balance=row['AOFIN'], 
            interest_rate=row['ARATE'], 
            original_term=row['AOTRM'], 
            calculation_start_date=row['ACODTDT'],
            next_payment_date=row['AUD3DT'], 
            cpr=row['CPR'], 
            cdr=row['CDR'], 
            severity=row['Severity'],
            price=row['PRICE'],
            arpay=row['ARPAY'],
            xirr_calc=True,
            coupon_mult=row['COUPON_MULT']  # Add this line
        )
    else: 
        priceNa += 1
        loan_yield = None
        
    yields.append(loan_yield)

loan_dataset['Original Yields'] = yields

24090it [00:26, 898.47it/s]


In [9]:
prices = []
cpr_list = []
cdr_list = []
severity_list = []
coupon_mult_list = []  # Add this line
priceNa = 0
for index, row in tqdm(loan_dataset.iterrows()):
    if pd.notna(row['PRICE']):
        npv, cpr, cdr, severity, coupon_mult = get_cash_flows(  # Update unpacking
            apmt1=row['APMT1'],
            rempmts=row['REMPMTS'],
            original_balance=row['AOFIN'],
            unpaid_balance=row['ANETBAL'],
            interest_rate=row['ARATE'],
            original_term=row['AOTRM'],
            calculation_start_date=row['ABKDTDT'],
            next_payment_date=row['ANXDTDT'],
            cpr=row['CPR'],
            cdr=row['CDR'],
            severity=row['Severity'],
            price=row['PRICE'],
            original_yield=row['Original Yields'],
            arpay=row['ARPAY'],
            contract_date=row['ACODTDT'],
            maturity_date=row['AMTDTDT'],
            coupon_mult=row['COUPON_MULT'],
            reference_date=row['ABKDTDT'] # Add this line
        )
        original_balance = row['ANETBAL'] if row['ANETBAL'] > 0 else 0
        price = npv*100/row['ANETBAL']
    else:
        priceNa += 1
        price = None
        
    prices.append(price)
    cpr_list.append(cpr)
    cdr_list.append(cdr)
    severity_list.append(severity)
    coupon_mult_list.append(coupon_mult)  # Add this line

loan_dataset['Trimmed CPR'] = cpr_list
loan_dataset['Trimmed CDR'] = cdr_list
loan_dataset['Trimmed Severity'] = severity_list
loan_dataset['Trimmed COUPON_MULT'] = coupon_mult_list  # Add this line
loan_dataset['New Price'] = prices

loan_dataset.to_excel('Original Yields and New Prices.xlsx', index=False)

467it [00:00, 754.04it/s]/tmp/ipykernel_1011198/166261007.py:29: RuntimeWarning: invalid value encountered in scalar divide
  price = npv*100/row['ANETBAL']
24090it [00:27, 874.93it/s]


In [10]:
from pandas import Timestamp

# Initialize lists for each monthboard
oct_dfs07 = []
orig_dfs07 = []
oct_dfs08 = []
orig_dfs08 = []
oct_dfs09 = []
orig_dfs09 = []
oct_dfs10 = []
orig_dfs10 = []
oct_dfs11 = []
orig_dfs11 = []
oct_dfs12 = []
orig_dfs12 = []

for _, row in tqdm(loan_dataset.iterrows()):
    if pd.notna(row['PRICE']):
        oct31_df, _, _ = get_cash_flows(
            apmt1=row['APMT1'],
            rempmts=row['REMPMTS'],
            original_balance=row['AOFIN'], 
            unpaid_balance=row['ANETBAL'],
            interest_rate=row['ARATE'],
            original_term=row['AOTRM'],
            calculation_start_date=row['ABKDTDT'],
            next_payment_date=row['ANXDTDT'],
            cpr=row['CPR'],
            cdr=row['CDR'],
            severity=row['Severity'],
            price=row['PRICE'],
            maturity_date=row['AMTDTDT'],
            contract_date=row['ACODTDT'],
            arpay=row['ARPAY'],
            output=True,
            coupon_mult=row['COUPON_MULT'],
            reference_date=row['ABKDTDT']  # Add this line
        )

        start_date = row['ACODTDT']
        next_payment_date = row['AUD3DT']

        # Add conditions for new monthboards
        if row['MONTHBOARD'] == 202307:
            if start_date < Timestamp('2023-07-01'):
                start_date = Timestamp('2023-07-01')
                if next_payment_date < Timestamp('2023-07-01'):
                    next_payment_date = Timestamp('2023-07-02')
        if row['MONTHBOARD'] == 202308:
            if start_date < Timestamp('2023-08-01'):
                start_date = Timestamp('2023-08-01')
                if next_payment_date < Timestamp('2023-08-01'):
                    next_payment_date = Timestamp('2023-08-02')
        if row['MONTHBOARD'] == 202309:
            if start_date < Timestamp('2023-09-01'):
                start_date = Timestamp('2023-09-01')
                if next_payment_date < Timestamp('2023-09-01'):
                    next_payment_date = Timestamp('2023-09-02')
        if row['MONTHBOARD'] == 202310:
            if start_date < Timestamp('2023-10-01'):
                start_date = Timestamp('2023-10-01')
                if next_payment_date < Timestamp('2023-10-01'):
                    next_payment_date = Timestamp('2023-10-02')
        if row['MONTHBOARD'] == 202311:
            if start_date < Timestamp('2023-11-01'):
                start_date = Timestamp('2023-11-01')
                if next_payment_date < Timestamp('2023-11-01'):
                    next_payment_date = Timestamp('2023-11-02')
        if row['MONTHBOARD'] == 202312:
            if start_date < Timestamp('2023-12-01'):
                start_date = Timestamp('2023-12-01')
                if next_payment_date < Timestamp('2023-12-01'):
                    next_payment_date = Timestamp('2023-12-02')

        origination_df, _, _ = get_cash_flows(
            apmt1=row['APMT1'],
            rempmts=row['AOTRM'],
            original_balance=row['AOFIN'],
            unpaid_balance=row['AOFIN'],
            interest_rate=row['ARATE'],
            original_term=row['AOTRM'],
            calculation_start_date=start_date,
            next_payment_date=next_payment_date, 
            cpr=row['CPR'],
            cdr=row['CDR'], 
            severity=row['Severity'],
            price=row['PRICE'],
            output=True,
            arpay=row['ARPAY'],
            coupon_mult=row['COUPON_MULT'],
            reference_date=row['ABKDTDT']  # Add this line
        )

        # Add conditions for new monthboards
        if row['MONTHBOARD'] == 202307:
            oct_dfs07.append(oct31_df)
            orig_dfs07.append(origination_df)
        if row['MONTHBOARD'] == 202308:
            oct_dfs08.append(oct31_df)
            orig_dfs08.append(origination_df)
        if row['MONTHBOARD'] == 202309:
            oct_dfs09.append(oct31_df)
            orig_dfs09.append(origination_df)
        if row['MONTHBOARD'] == 202310:
            oct_dfs10.append(oct31_df)
            orig_dfs10.append(origination_df)
        if row['MONTHBOARD'] == 202311:
            oct_dfs11.append(oct31_df)
            orig_dfs11.append(origination_df)
        if row['MONTHBOARD'] == 202312:
            oct_dfs12.append(oct31_df)
            orig_dfs12.append(origination_df)
    else:
        continue

print("Converting to Monthly")

def convert_to_monthly(df):
    # Convert the Date column to datetime
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Group by year and month, and then apply custom aggregations
    aggregations = {
        'UPB': 'max',
        'Performing UPB': 'max',
        'Gross Charge Offs': 'sum',
        'Net Loss': 'sum',
        'Recoveries': 'sum',
        'Scheduled Principal': 'sum',
        'Voluntary Prepayments': 'sum',
        'Principal Cash Flow': 'sum',
        'Interest Cash Flow': 'sum',
        'Total Cash Flow': 'sum'
    }
    
    # Group and aggregate
    df_grouped = df.groupby(df['Date'].dt.to_period("M")).agg(aggregations).reset_index()
    
    # Ensure chronological order
    df_grouped.sort_values('Date', inplace=True)
    
    # Convert the Date from Period to datetime (start of the period)
    df_grouped['Date'] = df_grouped['Date'].dt.to_timestamp()

    # Check for and insert missing months
    all_months = pd.date_range(start=df_grouped['Date'].min(), end=df_grouped['Date'].max(), freq='MS')
    missing_months = all_months.difference(df_grouped['Date'])
    
    # For each missing month, insert a row with values from the first row
    for missing_month in missing_months:
        # Find the previous month to the missing month in the dataset
        prev_month = missing_month - pd.DateOffset(months=1)
        if prev_month in df_grouped['Date'].values:
            prev_month_row = df_grouped.loc[df_grouped['Date'] == prev_month].copy()
            prev_month_row['Date'] = missing_month
            df_grouped = pd.concat([df_grouped, prev_month_row], ignore_index=True)
        else:
            # If the missing month is the first month, duplicate the first row with the new month
            first_row = df_grouped.iloc[0].copy()
            first_row['Date'] = missing_month
            df_grouped = pd.concat([df_grouped, pd.DataFrame([first_row])], ignore_index=True)

    # Sort again after inserting rows
    df_grouped.sort_values('Date', inplace=True)
    df_grouped.reset_index(drop=True, inplace=True)

    return df_grouped

def combine_dfs_by_month(dfs):
    for df in dfs:
        df['Date'] = pd.to_datetime(df['Date'])
    for df in dfs:
        df.set_index('Date', inplace=True)

    combined_df = pd.concat(dfs)

    monthly_sum = combined_df.resample('ME').sum()
    monthly_sum.reset_index(inplace=True)

    return monthly_sum

# Update lists to include new monthboards
dfs = [oct_dfs07, orig_dfs07, oct_dfs08, orig_dfs08, oct_dfs09, orig_dfs09,
       oct_dfs10, orig_dfs10, oct_dfs11, orig_dfs11, oct_dfs12, orig_dfs12]
names = ['oct_dfs07', 'orig_dfs07', 'oct_dfs08', 'orig_dfs08', 'oct_dfs09', 'orig_dfs09',
         'oct_dfs10', 'orig_dfs10', 'oct_dfs11', 'orig_dfs11', 'oct_dfs12', 'orig_dfs12']

combined_dfs = []
for df in tqdm(dfs):
    new_dfs = []
    for sub_df in df:
        new_dfs.append(convert_to_monthly(sub_df))
    combined_dfs.append(combine_dfs_by_month(new_dfs))

# Save to Excel
print("Saving to Excel")
for i, df in enumerate(dfs):
    combined_dfs[i].to_excel(f'monthboard_{names[i]}.xlsx', index=False)

24090it [01:06, 362.18it/s]


Converting to Monthly


100%|██████████| 12/12 [03:04<00:00, 15.38s/it]

Saving to Excel
